# Advanced Problems with Solutions: Python Objects and Classes

This notebook builds on the core ideas of **objects and classes**:

- creating classes with `class`
- class objects vs. instances
- `type(...)`
- `__name__`
- `__class__`
- `isinstance(...)`
- classes as callable objects
- `type` as the class of classes
- dynamic class creation with `type(name, bases, dict)`

The notebook intentionally contains **many examples, problems, predictions, debugging tasks, and solution cells**.

> Best-practice workflow: for each problem, first predict the result without running the solution, then test your prediction, then explain *why* Python behaves that way.

## 0. Warm-up Reference

A class definition creates a **class object**. Calling that class object usually creates an **instance**.

The expressions below are worth keeping mentally separate:

- `Person` → the class object
- `Person()` → a new instance of `Person`
- `type(Person)` → normally `type`
- `type(Person())` → `Person`
- `Person.__name__` → the class name as a string
- `obj.__class__` → the class used to create `obj`
- `isinstance(obj, Person)` → whether `obj` is an instance of `Person`

In [1]:
class Person:
    pass

p = Person()

print("Person:", Person)
print("p:", p)
print("type(Person):", type(Person))
print("type(p):", type(p))
print("Person.__name__:", Person.__name__)
print("p.__class__:", p.__class__)
print("type(p) is p.__class__:", type(p) is p.__class__)
print("isinstance(p, Person):", isinstance(p, Person))
print("isinstance(Person, type):", isinstance(Person, type))
print("isinstance(type, type):", isinstance(type, type))

Person: <class '__main__.Person'>
p: <__main__.Person object at 0x0000020FEBA58440>
type(Person): <class 'type'>
type(p): <class '__main__.Person'>
Person.__name__: Person
p.__class__: <class '__main__.Person'>
type(p) is p.__class__: True
isinstance(p, Person): True
isinstance(Person, type): True
isinstance(type, type): True


---

# Problem 1 — Predict the Object Relationships

Without running the code first, predict the values of `A` through `H`.

```python
class Device:
    pass

d1 = Device()
d2 = Device()

A = type(Device)
B = type(d1)
C = d1.__class__
D = type(d1) is d1.__class__
E = d1 is d2
F = type(d1) is type(d2)
G = isinstance(d2, Device)
H = isinstance(Device, type)
```

Explain every answer using the distinction between a **class object** and an **instance object**.

## Solution 1

In [2]:
class Device:
    pass

d1 = Device()
d2 = Device()

answers = {
    "A = type(Device)": type(Device),
    "B = type(d1)": type(d1),
    "C = d1.__class__": d1.__class__,
    "D = type(d1) is d1.__class__": type(d1) is d1.__class__,
    "E = d1 is d2": d1 is d2,
    "F = type(d1) is type(d2)": type(d1) is type(d2),
    "G = isinstance(d2, Device)": isinstance(d2, Device),
    "H = isinstance(Device, type)": isinstance(Device, type),
}

for label, value in answers.items():
    print(f"{label:38} -> {value}")

A = type(Device)                       -> <class 'type'>
B = type(d1)                           -> <class '__main__.Device'>
C = d1.__class__                       -> <class '__main__.Device'>
D = type(d1) is d1.__class__           -> True
E = d1 is d2                           -> False
F = type(d1) is type(d2)               -> True
G = isinstance(d2, Device)             -> True
H = isinstance(Device, type)           -> True


### Why

- `Device` is itself an object; its type is `type`.
- `d1` and `d2` are distinct instances of `Device`.
- Both instances have the same class.
- `type(d1)` and `d1.__class__` identify the same class object.
- `is` compares object identity, so `d1 is d2` is `False`.

---

# Problem 2 — Build an Introspection Report

Write a function:

```python
def describe_object(obj):
    ...
```

It should return a dictionary containing:

1. the object itself
2. its type
3. its class via `__class__`
4. whether `type(obj) is obj.__class__`
5. whether the object is itself a class object
6. whether the object is callable
7. its `__name__` if one exists, otherwise `None`

Test it with:

- `42`
- `"python"`
- `Person`
- `Person()`
- `type`
- `len`

## Solution 2

In [3]:
def describe_object(obj):
    return {
        "object": obj,
        "type": type(obj),
        "__class__": obj.__class__,
        "type_is___class__": type(obj) is obj.__class__,
        "is_class_object": isinstance(obj, type),
        "callable": callable(obj),
        "__name__": getattr(obj, "__name__", None),
    }

class Person:
    pass

samples = [42, "python", Person, Person(), type, len]

for sample in samples:
    print("=" * 70)
    for key, value in describe_object(sample).items():
        print(f"{key:20} -> {value}")

object               -> 42
type                 -> <class 'int'>
__class__            -> <class 'int'>
type_is___class__    -> True
is_class_object      -> False
callable             -> False
__name__             -> None
object               -> python
type                 -> <class 'str'>
__class__            -> <class 'str'>
type_is___class__    -> True
is_class_object      -> False
callable             -> False
__name__             -> None
object               -> <class '__main__.Person'>
type                 -> <class 'type'>
__class__            -> <class 'type'>
type_is___class__    -> True
is_class_object      -> True
callable             -> True
__name__             -> Person
object               -> <__main__.Person object at 0x0000020FFBA3E900>
type                 -> <class '__main__.Person'>
__class__            -> <class '__main__.Person'>
type_is___class__    -> True
is_class_object      -> False
callable             -> False
__name__             -> None
object             

### Best-practice note

Use `getattr(obj, "__name__", None)` when an attribute is optional. It avoids writing a manual `try/except` just to check whether the attribute exists.

---

# Problem 3 — Class Names Are Data

Create three classes named `Alpha`, `Beta`, and `Gamma`.

Write a function:

```python
def class_names(classes):
    ...
```

that accepts a list of class objects and returns their names as strings.

Then verify that every item supplied to the function is actually a class object. Raise `TypeError` if a non-class value is supplied.

## Solution 3

In [4]:
class Alpha:
    pass

class Beta:
    pass

class Gamma:
    pass

def class_names(classes):
    result = []

    for cls in classes:
        if not isinstance(cls, type):
            raise TypeError(f"Expected a class object, got {type(cls).__name__}")
        result.append(cls.__name__)

    return result

print(class_names([Alpha, Beta, Gamma]))

['Alpha', 'Beta', 'Gamma']


In [5]:
# Failure example
try:
    class_names([Alpha, 123, Gamma])
except TypeError as exc:
    print("Caught:", exc)

Caught: Expected a class object, got int


---

# Problem 4 — Class Object or Instance?

Implement:

```python
def classify(value):
    ...
```

Return one of these strings:

- `"class object"`
- `"instance of <ClassName>"`

Examples:

```python
classify(Person)      # "class object"
classify(Person())    # "instance of Person"
classify(10)          # "instance of int"
```

Do not hard-code any class names.

## Solution 4

In [6]:
def classify(value):
    if isinstance(value, type):
        return "class object"
    return f"instance of {type(value).__name__}"

class Person:
    pass

for value in [Person, Person(), 10, "hello", type, object]:
    print(f"{value!r:40} -> {classify(value)}")

<class '__main__.Person'>                -> class object
<__main__.Person object at 0x0000020FFBA3EA50> -> instance of Person
10                                       -> instance of int
'hello'                                  -> instance of str
<class 'type'>                           -> class object
<class 'object'>                         -> class object


---

# Problem 5 — Why Are Classes Callable?

The source material notes that classes are callable.

Investigate this with:

```python
class Empty:
    pass
```

Answer:

1. Is `Empty` callable?
2. Is `Empty()` callable?
3. Is `type` callable?
4. Is `object` callable?
5. Is `42` callable?

Then write a helper that accepts a value and prints both `type(value)` and `callable(value)`.

## Solution 5

In [7]:
class Empty:
    pass

values = [Empty, Empty(), type, object, 42]

for value in values:
    print(
        f"value={value!r:45} "
        f"type={type(value)!r:30} "
        f"callable={callable(value)}"
    )

value=<class '__main__.Empty'>                      type=<class 'type'>                 callable=True
value=<__main__.Empty object at 0x0000020FFBA3EA50> type=<class '__main__.Empty'>       callable=False
value=<class 'type'>                                type=<class 'type'>                 callable=True
value=<class 'object'>                              type=<class 'type'>                 callable=True
value=42                                            type=<class 'int'>                  callable=False


### Explanation

A class object is callable because calling it initiates instance creation. This behavior is ultimately provided by the machinery of `type`, which implements class call behavior.

---

# Problem 6 — Identity vs. Type Equality

Consider:

```python
class Token:
    pass

a = Token()
b = Token()
c = a
```

Predict all results before running them:

```python
a is b
a is c
type(a) is type(b)
a.__class__ is b.__class__
isinstance(a, Token)
isinstance(Token, Token)
isinstance(Token, type)
```

Explain why `isinstance(Token, Token)` is different from `isinstance(Token, type)`.

## Solution 6

In [8]:
class Token:
    pass

a = Token()
b = Token()
c = a

expressions = [
    ("a is b", a is b),
    ("a is c", a is c),
    ("type(a) is type(b)", type(a) is type(b)),
    ("a.__class__ is b.__class__", a.__class__ is b.__class__),
    ("isinstance(a, Token)", isinstance(a, Token)),
    ("isinstance(Token, Token)", isinstance(Token, Token)),
    ("isinstance(Token, type)", isinstance(Token, type)),
]

for expression, result in expressions:
    print(f"{expression:35} -> {result}")

a is b                              -> False
a is c                              -> True
type(a) is type(b)                  -> True
a.__class__ is b.__class__          -> True
isinstance(a, Token)                -> True
isinstance(Token, Token)            -> False
isinstance(Token, type)             -> True


### Key idea

`Token` is a class object. It is not normally an instance of itself. Its type is `type`, so `isinstance(Token, type)` is true.

---

# Problem 7 — Build a Safe Factory

Write:

```python
def make_instance(cls):
    ...
```

Requirements:

- `cls` must be a class object.
- If it is not a class object, raise `TypeError`.
- Otherwise, call it with no arguments and return the created instance.

Test it with user-defined classes and built-in classes.

## Solution 7

In [9]:
def make_instance(cls):
    if not isinstance(cls, type):
        raise TypeError("make_instance expects a class object")
    return cls()

class Marker:
    pass

examples = [Marker, list, dict, set]

for cls in examples:
    instance = make_instance(cls)
    print(
        f"class={cls.__name__:10} "
        f"instance={instance!r:12} "
        f"type={type(instance).__name__}"
    )

class=Marker     instance=<__main__.Marker object at 0x0000020FFBA3EE40> type=Marker
class=list       instance=[]           type=list
class=dict       instance={}           type=dict
class=set        instance=set()        type=set


In [10]:
# Incorrect input
try:
    make_instance(Marker())
except TypeError as exc:
    print("Caught:", exc)

Caught: make_instance expects a class object


---

# Problem 8 — A Class Registry

Create a registry that stores class objects by class name.

Required API:

```python
registry = {}

def register(cls):
    ...

def create(name):
    ...
```

Rules:

- `register(cls)` accepts only class objects.
- Use `cls.__name__` as the key.
- `create(name)` creates and returns a new instance of the registered class.
- Raise a useful error for unknown names.

Register at least four classes.

## Solution 8

In [11]:
registry = {}

def register(cls):
    if not isinstance(cls, type):
        raise TypeError("Only class objects can be registered")
    registry[cls.__name__] = cls

def create(name):
    if name not in registry:
        available = ", ".join(sorted(registry)) or "<none>"
        raise KeyError(f"Unknown class {name!r}. Available: {available}")
    return registry[name]()

class Red:
    pass

class Green:
    pass

class Blue:
    pass

class Yellow:
    pass

for cls in [Red, Green, Blue, Yellow]:
    register(cls)

print("Registry:", registry)

for name in ["Red", "Green", "Blue", "Yellow"]:
    obj = create(name)
    print(name, "->", obj, "| type:", type(obj).__name__)

Registry: {'Red': <class '__main__.Red'>, 'Green': <class '__main__.Green'>, 'Blue': <class '__main__.Blue'>, 'Yellow': <class '__main__.Yellow'>}
Red -> <__main__.Red object at 0x0000020FFBA3ECF0> | type: Red
Green -> <__main__.Green object at 0x0000020FFBA3EE40> | type: Green
Blue -> <__main__.Blue object at 0x0000020FFBA3ECF0> | type: Blue
Yellow -> <__main__.Yellow object at 0x0000020FFBA3EE40> | type: Yellow


---

# Problem 9 — Validate Object/Class Consistency

Write:

```python
def validate_instance(obj, expected_class):
    ...
```

It should:

1. verify `expected_class` is a class object
2. return `True` if `obj` is an instance of that class
3. return `False` otherwise

Then write a stricter version:

```python
def require_instance(obj, expected_class):
    ...
```

that raises `TypeError` when the object is not an instance of the expected class.

## Solution 9

In [12]:
def validate_instance(obj, expected_class):
    if not isinstance(expected_class, type):
        raise TypeError("expected_class must be a class object")
    return isinstance(obj, expected_class)

def require_instance(obj, expected_class):
    if not validate_instance(obj, expected_class):
        raise TypeError(
            f"Expected instance of {expected_class.__name__}, "
            f"got {type(obj).__name__}"
        )
    return obj

class Customer:
    pass

customer = Customer()

print(validate_instance(customer, Customer))
print(validate_instance("not a customer", Customer))

try:
    require_instance("not a customer", Customer)
except TypeError as exc:
    print("Caught:", exc)

True
False
Caught: Expected instance of Customer, got str


---

# Problem 10 — `type(obj)` vs. `obj.__class__`

Construct a table for the following values:

```python
0
3.14
"abc"
[]
{}
set()
lambda x: x
Person
Person()
type
```

For every value, collect:

- `repr(value)`
- `type(value).__name__`
- `value.__class__.__name__`
- whether `type(value) is value.__class__`
- whether it is callable
- whether it is a class object

## Solution 10

In [13]:
class Person:
    pass

values = [
    0,
    3.14,
    "abc",
    [],
    {},
    set(),
    lambda x: x,
    Person,
    Person(),
    type,
]

rows = []

for value in values:
    rows.append({
        "repr": repr(value),
        "type": type(value).__name__,
        "__class__": value.__class__.__name__,
        "same_class_object": type(value) is value.__class__,
        "callable": callable(value),
        "is_class_object": isinstance(value, type),
    })

for row in rows:
    print(row)

{'repr': '0', 'type': 'int', '__class__': 'int', 'same_class_object': True, 'callable': False, 'is_class_object': False}
{'repr': '3.14', 'type': 'float', '__class__': 'float', 'same_class_object': True, 'callable': False, 'is_class_object': False}
{'repr': "'abc'", 'type': 'str', '__class__': 'str', 'same_class_object': True, 'callable': False, 'is_class_object': False}
{'repr': '[]', 'type': 'list', '__class__': 'list', 'same_class_object': True, 'callable': False, 'is_class_object': False}
{'repr': '{}', 'type': 'dict', '__class__': 'dict', 'same_class_object': True, 'callable': False, 'is_class_object': False}
{'repr': 'set()', 'type': 'set', '__class__': 'set', 'same_class_object': True, 'callable': False, 'is_class_object': False}
{'repr': '<function <lambda> at 0x0000020FFBAE0CC0>', 'type': 'function', '__class__': 'function', 'same_class_object': True, 'callable': True, 'is_class_object': False}
{'repr': "<class '__main__.Person'>", 'type': 'type', '__class__': 'type', 'same_cl

---

# Problem 11 — Debug This Incorrect Type Checker

A developer writes:

```python
def is_person(obj):
    return type(obj).__name__ == "Person"
```

Explain at least **three problems** with this design.

Then rewrite it using Python's intended type-checking mechanism.

## Solution 11

Problems with comparing class-name strings:

1. It compares text rather than actual class identity/relationships.
2. Two unrelated classes can share the same `__name__`.
3. Renaming the class silently breaks the check.
4. It bypasses Python's normal instance-checking protocol.
5. It becomes especially fragile once inheritance enters the picture.

Use `isinstance`.

In [14]:
class Person:
    pass

def is_person(obj):
    return isinstance(obj, Person)

print(is_person(Person()))
print(is_person("Person"))
print(is_person(Person))

True
False
False


---

# Problem 12 — Two Different Classes with the Same Name

Use `type(name, bases, dict)` to create **two distinct classes** that both have the name `"Duplicate"`.

Show that:

- their `__name__` values are equal
- the class objects are not identical
- an instance of the first is not an instance of the second

This demonstrates why checking only `__name__` is unsafe.

## Solution 12

In [15]:
DuplicateA = type("Duplicate", (), {})
DuplicateB = type("Duplicate", (), {})

a = DuplicateA()
b = DuplicateB()

print("DuplicateA.__name__:", DuplicateA.__name__)
print("DuplicateB.__name__:", DuplicateB.__name__)
print("Same name:", DuplicateA.__name__ == DuplicateB.__name__)
print("Same class object:", DuplicateA is DuplicateB)
print("type(a) is DuplicateA:", type(a) is DuplicateA)
print("type(a) is DuplicateB:", type(a) is DuplicateB)
print("isinstance(a, DuplicateA):", isinstance(a, DuplicateA))
print("isinstance(a, DuplicateB):", isinstance(a, DuplicateB))

DuplicateA.__name__: Duplicate
DuplicateB.__name__: Duplicate
Same name: True
Same class object: False
type(a) is DuplicateA: True
type(a) is DuplicateB: False
isinstance(a, DuplicateA): True
isinstance(a, DuplicateB): False


---

# Problem 13 — Dynamic Class Creation

The `type` built-in can be used in two ways:

```python
type(obj)
type(name, bases, namespace)
```

Create a class dynamically that is equivalent to:

```python
class Dynamic:
    category = "generated"
```

Then instantiate it and inspect:

- the class
- its name
- its type
- the instance type
- the instance's class
- the `category` attribute

## Solution 13

In [16]:
Dynamic = type(
    "Dynamic",
    (),
    {
        "category": "generated",
    },
)

obj = Dynamic()

print("Dynamic:", Dynamic)
print("Dynamic.__name__:", Dynamic.__name__)
print("type(Dynamic):", type(Dynamic))
print("type(obj):", type(obj))
print("obj.__class__:", obj.__class__)
print("obj.category:", obj.category)
print("isinstance(obj, Dynamic):", isinstance(obj, Dynamic))

Dynamic: <class '__main__.Dynamic'>
Dynamic.__name__: Dynamic
type(Dynamic): <class 'type'>
type(obj): <class '__main__.Dynamic'>
obj.__class__: <class '__main__.Dynamic'>
obj.category: generated
isinstance(obj, Dynamic): True


---

# Problem 14 — Add a Method Dynamically

Create this behavior without a `class` statement:

```python
class Greeter:
    def greet(self):
        return "Hello from Greeter"
```

Use `type(name, bases, namespace)`.

## Solution 14

In [17]:
def greet(self):
    return f"Hello from {self.__class__.__name__}"

Greeter = type(
    "Greeter",
    (),
    {
        "greet": greet,
    },
)

g = Greeter()

print(g.greet())
print(type(g))
print(g.__class__)
print(isinstance(g, Greeter))

Hello from Greeter
<class '__main__.Greeter'>
<class '__main__.Greeter'>
True


---

# Problem 15 — Dynamic Class Factory

Write:

```python
def make_empty_class(name):
    ...
```

It should dynamically create and return a new class with the supplied name.

Then create five classes in a loop:

```text
Model1
Model2
Model3
Model4
Model5
```

Create one instance of each.

## Solution 15

In [18]:
def make_empty_class(name):
    if not isinstance(name, str):
        raise TypeError("name must be a string")
    if not name:
        raise ValueError("name cannot be empty")
    return type(name, (), {})

classes = [make_empty_class(f"Model{i}") for i in range(1, 6)]
instances = [cls() for cls in classes]

for cls, obj in zip(classes, instances):
    print(
        f"class={cls!r:30} "
        f"name={cls.__name__:8} "
        f"instance_type={type(obj).__name__}"
    )

class=<class '__main__.Model1'>      name=Model1   instance_type=Model1
class=<class '__main__.Model2'>      name=Model2   instance_type=Model2
class=<class '__main__.Model3'>      name=Model3   instance_type=Model3
class=<class '__main__.Model4'>      name=Model4   instance_type=Model4
class=<class '__main__.Model5'>      name=Model5   instance_type=Model5


---

# Problem 16 — Build a Configurable Dynamic Class

Write:

```python
def make_class(name, attributes):
    ...
```

Requirements:

- `name` must be a non-empty string.
- `attributes` must be a dictionary.
- Return a new class created with `type`.
- Do not mutate the original dictionary.

Test with:

```python
{"species": "cat", "legs": 4}
```

## Solution 16

In [19]:
def make_class(name, attributes):
    if not isinstance(name, str) or not name:
        raise ValueError("name must be a non-empty string")

    if not isinstance(attributes, dict):
        raise TypeError("attributes must be a dictionary")

    namespace = dict(attributes)
    return type(name, (), namespace)

attributes = {"species": "cat", "legs": 4}
CatInfo = make_class("CatInfo", attributes)

cat = CatInfo()

print(CatInfo.__name__)
print(cat.species)
print(cat.legs)
print("Original dictionary:", attributes)

CatInfo
cat
4
Original dictionary: {'species': 'cat', 'legs': 4}


---

# Problem 17 — Compare `class` Syntax with `type(...)`

Create one class using normal syntax and one using `type(...)`.

They should both expose:

```python
label = "example"
```

Then compare:

- names
- types
- callability
- instance creation
- `isinstance`
- attribute access

## Solution 17

In [20]:
class Normal:
    label = "example"

DynamicEquivalent = type(
    "DynamicEquivalent",
    (),
    {"label": "example"},
)

normal_obj = Normal()
dynamic_obj = DynamicEquivalent()

for cls, obj in [
    (Normal, normal_obj),
    (DynamicEquivalent, dynamic_obj),
]:
    print("=" * 60)
    print("Class:", cls)
    print("Class name:", cls.__name__)
    print("Class type:", type(cls))
    print("Class callable:", callable(cls))
    print("Instance:", obj)
    print("Instance type:", type(obj))
    print("isinstance:", isinstance(obj, cls))
    print("label:", obj.label)

Class: <class '__main__.Normal'>
Class name: Normal
Class type: <class 'type'>
Class callable: True
Instance: <__main__.Normal object at 0x0000020FFBA3F380>
Instance type: <class '__main__.Normal'>
isinstance: True
label: example
Class: <class '__main__.DynamicEquivalent'>
Class name: DynamicEquivalent
Class type: <class 'type'>
Class callable: True
Instance: <__main__.DynamicEquivalent object at 0x0000020FFBA3FE00>
Instance type: <class '__main__.DynamicEquivalent'>
isinstance: True
label: example


---

# Problem 18 — Instance Counter Without `__init__`

Stay close to the current topic and avoid relying on constructors.

Create a function `build(cls, n)` that calls a class object `n` times and returns the resulting list of instances.

Then prove:

- every item is an instance of `cls`
- each item has the same type
- each item is a distinct object

## Solution 18

In [21]:
def build(cls, n):
    if not isinstance(cls, type):
        raise TypeError("cls must be a class object")
    if not isinstance(n, int) or n < 0:
        raise ValueError("n must be a non-negative integer")
    return [cls() for _ in range(n)]

class Event:
    pass

events = build(Event, 5)

print("Count:", len(events))
print("All instances:", all(isinstance(x, Event) for x in events))
print("All same type:", all(type(x) is Event for x in events))
print("Unique identities:", len({id(x) for x in events}) == len(events))

for index, event in enumerate(events, start=1):
    print(index, event, id(event))

Count: 5
All instances: True
All same type: True
Unique identities: True
1 <__main__.Event object at 0x0000020FFBB34050> 2267670593616
2 <__main__.Event object at 0x0000020FEB890A50> 2267399391824
3 <__main__.Event object at 0x0000020FFBAD4550> 2267670201680
4 <__main__.Event object at 0x0000020FFBA88770> 2267669890928
5 <__main__.Event object at 0x0000020FFBA888A0> 2267669891232


---

# Problem 19 — Detect Class Objects in Mixed Data

Given:

```python
items = [
    int,
    10,
    str,
    "hello",
    list,
    [],
    Person,
    Person(),
    type,
]
```

Split them into:

```python
class_objects = [...]
non_class_objects = [...]
```

Use `isinstance(value, type)`.

## Solution 19

In [22]:
class Person:
    pass

items = [
    int,
    10,
    str,
    "hello",
    list,
    [],
    Person,
    Person(),
    type,
]

class_objects = [value for value in items if isinstance(value, type)]
non_class_objects = [value for value in items if not isinstance(value, type)]

print("CLASS OBJECTS")
for value in class_objects:
    print(" ", value)

print("\nNON-CLASS OBJECTS")
for value in non_class_objects:
    print(" ", repr(value))

CLASS OBJECTS
  <class 'int'>
  <class 'str'>
  <class 'list'>
  <class '__main__.Person'>
  <class 'type'>

NON-CLASS OBJECTS
  10
  'hello'
  []


---

# Problem 20 — A Generic Batch Instantiator

Write:

```python
def instantiate_all(classes):
    ...
```

It must:

- accept an iterable of class objects
- reject any non-class entry with a clear error
- return one new instance of each class

Test with both user-defined and built-in classes.

## Solution 20

In [23]:
def instantiate_all(classes):
    instances = []

    for index, cls in enumerate(classes):
        if not isinstance(cls, type):
            raise TypeError(
                f"Item at index {index} is not a class object: {cls!r}"
            )
        instances.append(cls())

    return instances

class A:
    pass

class B:
    pass

classes = [A, B, list, dict, set]
objects = instantiate_all(classes)

for cls, obj in zip(classes, objects):
    print(
        f"{cls.__name__:10} -> "
        f"{obj!r:12} | "
        f"isinstance={isinstance(obj, cls)}"
    )

A          -> <__main__.A object at 0x0000020FFBB342F0> | isinstance=True
B          -> <__main__.B object at 0x0000020FFBB34440> | isinstance=True
list       -> []           | isinstance=True
dict       -> {}           | isinstance=True
set        -> set()        | isinstance=True


---

# Problem 21 — Debugging: Accidental Instance Instead of Class

A program stores this:

```python
registry["Person"] = Person()
```

but later does:

```python
registry["Person"]()
```

Explain the bug.

Fix the registry so it stores class objects rather than instances.

## Solution 21

In [24]:
class Person:
    pass

# Wrong
bad_registry = {"Person": Person()}

print("Stored in bad_registry:", bad_registry["Person"])
print("Callable?", callable(bad_registry["Person"]))

# Correct
good_registry = {"Person": Person}

print("\nStored in good_registry:", good_registry["Person"])
print("Callable?", callable(good_registry["Person"]))

new_person = good_registry["Person"]()
print("Created:", new_person)
print("Type:", type(new_person))

Stored in bad_registry: <__main__.Person object at 0x0000020FFBB34590>
Callable? False

Stored in good_registry: <class '__main__.Person'>
Callable? True
Created: <__main__.Person object at 0x0000020FFBAD4690>
Type: <class '__main__.Person'>


---

# Problem 22 — Debugging: Confusing `type` with a Class Name String

Why is this a poor test?

```python
if type(obj).__name__ == expected_name:
    ...
```

Rewrite the API so the caller supplies a **class object** instead of a string.

## Solution 22

In [25]:
def matches_expected_class(obj, expected_class):
    if not isinstance(expected_class, type):
        raise TypeError("expected_class must be a class object")
    return isinstance(obj, expected_class)

class Report:
    pass

r = Report()

print(matches_expected_class(r, Report))
print(matches_expected_class("text", Report))

True
False


---

# Problem 23 — What Exactly Is `type`?

Evaluate and explain:

```python
type(type)
isinstance(type, type)
type.__class__
type(type) is type
```

Then compare the results with a user-defined class `Sample`.

## Solution 23

In [26]:
class Sample:
    pass

print("type(type):", type(type))
print("isinstance(type, type):", isinstance(type, type))
print("type.__class__:", type.__class__)
print("type(type) is type:", type(type) is type)

print("\n--- Sample ---")
print("type(Sample):", type(Sample))
print("isinstance(Sample, type):", isinstance(Sample, type))
print("Sample.__class__:", Sample.__class__)
print("type(Sample) is type:", type(Sample) is type)

type(type): <class 'type'>
isinstance(type, type): True
type.__class__: <class 'type'>
type(type) is type: True

--- Sample ---
type(Sample): <class 'type'>
isinstance(Sample, type): True
Sample.__class__: <class 'type'>
type(Sample) is type: True


### Interpretation

A normal user-defined class is an instance of `type`. Python's own `type` object is also an instance of `type`, which produces the unusual self-referential relationship:

```python
type(type) is type
```

---

# Problem 24 — Inspect `type` Without Dumping Everything

Instead of calling `help(type)` and reading a very large output, write code that checks whether these attributes exist:

```text
__call__
__new__
__init__
__repr__
__setattr__
__getattribute__
__subclasses__
mro
```

Use `hasattr`.

## Solution 24

In [27]:
names = [
    "__call__",
    "__new__",
    "__init__",
    "__repr__",
    "__setattr__",
    "__getattribute__",
    "__subclasses__",
    "mro",
]

for name in names:
    print(f"{name:20} -> {hasattr(type, name)}")

__call__             -> True
__new__              -> True
__init__             -> True
__repr__             -> True
__setattr__          -> True
__getattribute__     -> True
__subclasses__       -> True
mro                  -> True


---

# Problem 25 — Class Metadata Report

Write:

```python
def describe_class(cls):
    ...
```

Return:

- class name
- class object
- type of class object
- whether it is callable
- whether it is an instance of `type`

Reject non-class values.

## Solution 25

In [28]:
def describe_class(cls):
    if not isinstance(cls, type):
        raise TypeError("describe_class expects a class object")

    return {
        "name": cls.__name__,
        "class_object": cls,
        "type_of_class": type(cls),
        "callable": callable(cls),
        "instance_of_type": isinstance(cls, type),
    }

class Invoice:
    pass

for cls in [Invoice, int, str, list, type]:
    print(describe_class(cls))

{'name': 'Invoice', 'class_object': <class '__main__.Invoice'>, 'type_of_class': <class 'type'>, 'callable': True, 'instance_of_type': True}
{'name': 'int', 'class_object': <class 'int'>, 'type_of_class': <class 'type'>, 'callable': True, 'instance_of_type': True}
{'name': 'str', 'class_object': <class 'str'>, 'type_of_class': <class 'type'>, 'callable': True, 'instance_of_type': True}
{'name': 'list', 'class_object': <class 'list'>, 'type_of_class': <class 'type'>, 'callable': True, 'instance_of_type': True}
{'name': 'type', 'class_object': <class 'type'>, 'type_of_class': <class 'type'>, 'callable': True, 'instance_of_type': True}


---

# Problem 26 — Challenge: Mini Object Explorer

Write an interactive-style utility:

```python
def explore(*values):
    ...
```

For each value print:

- `repr(value)`
- `type(value)`
- `value.__class__`
- `type(value) is value.__class__`
- `callable(value)`
- `isinstance(value, type)`
- `getattr(value, "__name__", None)`

Use column-aligned output.

## Solution 26

In [29]:
def explore(*values):
    header = (
        f"{'repr':<28}"
        f"{'type':<18}"
        f"{'__class__':<18}"
        f"{'same?':<8}"
        f"{'call?':<8}"
        f"{'class?':<8}"
        f"{'__name__'}"
    )
    print(header)
    print("-" * len(header))

    for value in values:
        value_repr = repr(value)
        if len(value_repr) > 25:
            value_repr = value_repr[:22] + "..."

        print(
            f"{value_repr:<28}"
            f"{type(value).__name__:<18}"
            f"{value.__class__.__name__:<18}"
            f"{str(type(value) is value.__class__):<8}"
            f"{str(callable(value)):<8}"
            f"{str(isinstance(value, type)):<8}"
            f"{getattr(value, '__name__', None)}"
        )

class Example:
    pass

explore(
    1,
    2.5,
    "abc",
    [],
    {},
    Example,
    Example(),
    type,
    object,
    len,
)

repr                        type              __class__         same?   call?   class?  __name__
------------------------------------------------------------------------------------------------
1                           int               int               True    False   False   None
2.5                         float             float             True    False   False   None
'abc'                       str               str               True    False   False   None
[]                          list              list              True    False   False   None
{}                          dict              dict              True    False   False   None
<class '__main__.Examp...   type              type              True    True    True    Example
<__main__.Example obje...   Example           Example           True    False   False   None
<class 'type'>              type              type              True    True    True    type
<class 'object'>            type              type         

---

# Problem 27 — Challenge: Generate Classes from Names

Given:

```python
names = ["North", "South", "East", "West"]
```

Create a dictionary:

```python
{
    "North": <class ...>,
    "South": <class ...>,
    ...
}
```

using dynamic class creation.

Then instantiate every class.

## Solution 27

In [30]:
names = ["North", "South", "East", "West"]

classes = {
    name: type(name, (), {})
    for name in names
}

instances = {
    name: cls()
    for name, cls in classes.items()
}

for name in names:
    cls = classes[name]
    obj = instances[name]

    print(
        f"name={name:6} "
        f"class_name={cls.__name__:6} "
        f"object_type={type(obj).__name__:6} "
        f"valid={isinstance(obj, cls)}"
    )

name=North  class_name=North  object_type=North  valid=True
name=South  class_name=South  object_type=South  valid=True
name=East   class_name=East   object_type=East   valid=True
name=West   class_name=West   object_type=West   valid=True


---

# Problem 28 — Challenge: Dynamic Method Per Class

Create three classes dynamically:

```text
Adder
Multiplier
Power
```

Each class should provide an `operation(a, b)` method with different behavior:

- `Adder` → `a + b`
- `Multiplier` → `a * b`
- `Power` → `a ** b`

Then instantiate and test all three.

## Solution 28

In [31]:
def add(self, a, b):
    return a + b

def multiply(self, a, b):
    return a * b

def power(self, a, b):
    return a ** b

Adder = type("Adder", (), {"operation": add})
Multiplier = type("Multiplier", (), {"operation": multiply})
Power = type("Power", (), {"operation": power})

for cls in [Adder, Multiplier, Power]:
    obj = cls()
    print(
        f"{cls.__name__:12} "
        f"operation(2, 5) -> {obj.operation(2, 5)}"
    )

Adder        operation(2, 5) -> 7
Multiplier   operation(2, 5) -> 10
Power        operation(2, 5) -> 32


---

# Problem 29 — Challenge: Generic Constructor Table

Create at least ten class objects, then automatically build one instance of each class that can be called with no arguments.

Suggested classes:

```python
list, dict, set, tuple, str, int, float, bytes
```

plus your own empty classes.

Print:

- class name
- created instance
- instance type
- `isinstance` result

## Solution 29

In [32]:
class One:
    pass

class Two:
    pass

classes = [
    list,
    dict,
    set,
    tuple,
    str,
    int,
    float,
    bytes,
    One,
    Two,
]

for cls in classes:
    obj = cls()
    print(
        f"{cls.__name__:10} | "
        f"instance={obj!r:15} | "
        f"type={type(obj).__name__:10} | "
        f"valid={isinstance(obj, cls)}"
    )

list       | instance=[]              | type=list       | valid=True
dict       | instance={}              | type=dict       | valid=True
set        | instance=set()           | type=set        | valid=True
tuple      | instance=()              | type=tuple      | valid=True
str        | instance=''              | type=str        | valid=True
int        | instance=0               | type=int        | valid=True
float      | instance=0.0             | type=float      | valid=True
bytes      | instance=b''             | type=bytes      | valid=True
One        | instance=<__main__.One object at 0x0000020FFBA3F770> | type=One        | valid=True
Two        | instance=<__main__.Two object at 0x0000020FFBA3F620> | type=Two        | valid=True


---

# Problem 30 — Final Integrated Challenge: Plugin-Style Class Registry

Build a small class-based registry.

Requirements:

1. `register(cls)` stores a class object by `cls.__name__`.
2. Only class objects may be registered.
3. Duplicate names should raise an error rather than silently overwrite.
4. `available()` returns sorted registered names.
5. `create(name)` returns a new instance.
6. `create(name)` should raise a useful error for an unknown class.
7. Add a function `describe_registered()` that prints metadata for every registered class.
8. Register at least five user-defined classes.
9. Create at least one instance of each.
10. Verify every created object with `isinstance`.

This problem combines most ideas in the notebook.

## Solution 30

In [33]:
class ClassRegistry:
    def __init__(self):
        self._classes = {}

    def register(self, cls):
        if not isinstance(cls, type):
            raise TypeError(
                f"Expected a class object, got {type(cls).__name__}"
            )

        name = cls.__name__

        if name in self._classes:
            raise ValueError(f"Class name {name!r} is already registered")

        self._classes[name] = cls

    def available(self):
        return sorted(self._classes)

    def create(self, name):
        try:
            cls = self._classes[name]
        except KeyError:
            available = ", ".join(self.available()) or "<none>"
            raise KeyError(
                f"Unknown class {name!r}. Available: {available}"
            ) from None

        return cls()

    def describe_registered(self):
        for name in self.available():
            cls = self._classes[name]
            print(
                f"name={name:12} "
                f"type={type(cls).__name__:8} "
                f"callable={callable(cls)!s:5} "
                f"is_class={isinstance(cls, type)}"
            )

In [34]:
class EmailTask:
    pass

class BackupTask:
    pass

class CleanupTask:
    pass

class ExportTask:
    pass

class ReportTask:
    pass

registry = ClassRegistry()

for cls in [
    EmailTask,
    BackupTask,
    CleanupTask,
    ExportTask,
    ReportTask,
]:
    registry.register(cls)

print("Available:", registry.available())
print()

registry.describe_registered()
print()

created = []

for name in registry.available():
    obj = registry.create(name)
    created.append(obj)

    expected_class = obj.__class__

    print(
        f"created={obj!r:45} "
        f"type={type(obj).__name__:12} "
        f"verified={isinstance(obj, expected_class)}"
    )

Available: ['BackupTask', 'CleanupTask', 'EmailTask', 'ExportTask', 'ReportTask']

name=BackupTask   type=type     callable=True  is_class=True
name=CleanupTask  type=type     callable=True  is_class=True
name=EmailTask    type=type     callable=True  is_class=True
name=ExportTask   type=type     callable=True  is_class=True
name=ReportTask   type=type     callable=True  is_class=True

created=<__main__.BackupTask object at 0x0000020FFBA3F8C0> type=BackupTask   verified=True
created=<__main__.CleanupTask object at 0x0000020FFBA3F620> type=CleanupTask  verified=True
created=<__main__.EmailTask object at 0x0000020FFBA3FA10> type=EmailTask    verified=True
created=<__main__.ExportTask object at 0x0000020FFBA3FB60> type=ExportTask   verified=True
created=<__main__.ReportTask object at 0x0000020FFBB34D70> type=ReportTask   verified=True


In [35]:
# Duplicate registration test
try:
    registry.register(EmailTask)
except ValueError as exc:
    print("Duplicate test:", exc)

# Unknown class test
try:
    registry.create("MissingTask")
except KeyError as exc:
    print("Unknown-name test:", exc)

# Invalid registration test
try:
    registry.register(EmailTask())
except TypeError as exc:
    print("Invalid-object test:", exc)

Duplicate test: Class name 'EmailTask' is already registered
Unknown-name test: "Unknown class 'MissingTask'. Available: BackupTask, CleanupTask, EmailTask, ExportTask, ReportTask"
Invalid-object test: Expected a class object, got EmailTask


---

# Extra Practice Set — No Solutions Immediately Below

Try these before looking back at earlier solutions.

### Practice A
Write `is_class(value)` using only `isinstance`.

### Practice B
Write `same_runtime_class(a, b)` that returns whether both objects have the exact same runtime class object.

### Practice C
Write `class_name_of_instance(obj)` that returns the runtime class name.

### Practice D
Write `new_instances(cls, count)` with input validation.

### Practice E
Given a mixed list, keep only class objects.

### Practice F
Given a mixed list, keep only values that are callable.

### Practice G
Create ten empty classes dynamically in a loop.

### Practice H
Build a dictionary mapping dynamically generated class names to one instance each.

### Practice I
Create a class dynamically with attributes `x = 10`, `y = 20`, and a method `total(self)`.

### Practice J
Demonstrate with code that two classes can have the same `__name__` while still being different class objects.

# Extra Practice — Reference Solutions

In [36]:
# A
def is_class(value):
    return isinstance(value, type)

# B
def same_runtime_class(a, b):
    return type(a) is type(b)

# C
def class_name_of_instance(obj):
    return type(obj).__name__

# D
def new_instances(cls, count):
    if not isinstance(cls, type):
        raise TypeError("cls must be a class object")
    if not isinstance(count, int) or count < 0:
        raise ValueError("count must be a non-negative integer")
    return [cls() for _ in range(count)]

# E
mixed = [int, 1, str, "x", list, [], dict, {}]
only_classes = [x for x in mixed if isinstance(x, type)]

# F
only_callables = [x for x in mixed if callable(x)]

print("Only classes:", only_classes)
print("Only callables:", only_callables)

Only classes: [<class 'int'>, <class 'str'>, <class 'list'>, <class 'dict'>]
Only callables: [<class 'int'>, <class 'str'>, <class 'list'>, <class 'dict'>]


In [37]:
# G
generated_classes = [
    type(f"Generated{i}", (), {})
    for i in range(1, 11)
]

for cls in generated_classes:
    print(cls, cls.__name__, type(cls))

<class '__main__.Generated1'> Generated1 <class 'type'>
<class '__main__.Generated2'> Generated2 <class 'type'>
<class '__main__.Generated3'> Generated3 <class 'type'>
<class '__main__.Generated4'> Generated4 <class 'type'>
<class '__main__.Generated5'> Generated5 <class 'type'>
<class '__main__.Generated6'> Generated6 <class 'type'>
<class '__main__.Generated7'> Generated7 <class 'type'>
<class '__main__.Generated8'> Generated8 <class 'type'>
<class '__main__.Generated9'> Generated9 <class 'type'>
<class '__main__.Generated10'> Generated10 <class 'type'>


In [38]:
# H
instance_map = {
    cls.__name__: cls()
    for cls in generated_classes
}

for name, obj in instance_map.items():
    print(name, "->", obj)

Generated1 -> <__main__.Generated1 object at 0x0000020FFBB34EC0>
Generated2 -> <__main__.Generated2 object at 0x0000020FFBB35010>
Generated3 -> <__main__.Generated3 object at 0x0000020FFBB35160>
Generated4 -> <__main__.Generated4 object at 0x0000020FFBB352B0>
Generated5 -> <__main__.Generated5 object at 0x0000020FFBB35400>
Generated6 -> <__main__.Generated6 object at 0x0000020FFBB35550>
Generated7 -> <__main__.Generated7 object at 0x0000020FFBB356A0>
Generated8 -> <__main__.Generated8 object at 0x0000020FFBB357F0>
Generated9 -> <__main__.Generated9 object at 0x0000020FFBB35940>
Generated10 -> <__main__.Generated10 object at 0x0000020FFBB35A90>


In [39]:
# I
def total(self):
    return self.x + self.y

PointLike = type(
    "PointLike",
    (),
    {
        "x": 10,
        "y": 20,
        "total": total,
    },
)

point = PointLike()

print(point.x)
print(point.y)
print(point.total())

10
20
30


In [40]:
# J
First = type("SameName", (), {})
Second = type("SameName", (), {})

print("Names equal:", First.__name__ == Second.__name__)
print("Same class object:", First is Second)

first_obj = First()

print("isinstance(first_obj, First):", isinstance(first_obj, First))
print("isinstance(first_obj, Second):", isinstance(first_obj, Second))

Names equal: True
Same class object: False
isinstance(first_obj, First): True
isinstance(first_obj, Second): False


---

# Rapid-Fire Prediction Drill

Predict each result before executing.

In [41]:
class X:
    pass

x = X()

checks = [
    ("type(X) is type", type(X) is type),
    ("type(x) is X", type(x) is X),
    ("x.__class__ is X", x.__class__ is X),
    ("type(x) is x.__class__", type(x) is x.__class__),
    ("isinstance(x, X)", isinstance(x, X)),
    ("isinstance(X, type)", isinstance(X, type)),
    ("isinstance(X, X)", isinstance(X, X)),
    ("callable(X)", callable(X)),
    ("callable(x)", callable(x)),
    ("type(type) is type", type(type) is type),
    ("isinstance(type, type)", isinstance(type, type)),
]

for expression, result in checks:
    print(f"{expression:32} -> {result}")

type(X) is type                  -> True
type(x) is X                     -> True
x.__class__ is X                 -> True
type(x) is x.__class__           -> True
isinstance(x, X)                 -> True
isinstance(X, type)              -> True
isinstance(X, X)                 -> False
callable(X)                      -> True
callable(x)                      -> False
type(type) is type               -> True
isinstance(type, type)           -> True


# Best-Practice Summary

1. Use `isinstance(obj, SomeClass)` for normal instance checks.
2. Use `isinstance(value, type)` when you specifically need to verify that a value is a class object.
3. Do not use class-name strings as a substitute for actual class objects.
4. Remember that a class is itself an object.
5. Calling a class creates an instance when the class supports the supplied arguments.
6. `type(obj)` and `obj.__class__` normally identify the same runtime class.
7. `__name__` is useful for display and registries, but names alone do not prove class identity.
8. Validate inputs in reusable helpers and produce informative error messages.
9. Prefer small functions with one clear responsibility.
10. When learning introspection, predict first, execute second, explain third.

# Optional Exploration

Run these one at a time and inspect the output:

```python
help(type)
dir(type)
dir(Person)
dir(Person())
type.__dict__.keys()
Person.__dict__.keys()
```

These can produce a lot of output, so they are intentionally not executed automatically in this notebook.